## Match Profeel With SDES Stock

**Purpose**
Aligns Profeel categories with SDES dimensions and produces a consistent final Profeel stock file for model inputs.

**Primary inputs**
- `data/derived/profeel/building_stock_profeel_std.csv`
- `data/derived/building_stock/building_stock_sdes2018_aggregated.csv`

**Primary outputs**
- `data/derived/profeel/building_stock_profeel_final.csv`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Step 4: Profeel-SDES Data Matching

**Purpose:** Merge SDES and Profeel datasets by matching on Housing type, Performance, Energy, Heating system.

**Inputs:** SDES aggregated stock + Profeel building characteristics.

**Outputs:** building_stock_profeel_final.csv (35.17M dwellings).


In [ ]:
import os
import pandas as pd


# Merging SDES database with Profeel to get households information
The method is to match the sdes archetypes to a Profeel one according to the values for the three index levels: Housing types, Energy performance, heating energy.
And then we calculate among each group the weight of the sdes archetypes in terms of their sdes's size estimate. Finally, this weights allow us to get the size of each lines in terms of Profeel stock.

The merging rule is to conserve, from the database of origin (SDES), the proportions of each household characteristics among a group of buildings which have same basic building characteristics.


In [ ]:
profeel = pd.read_csv(os.path.join('data', 'derived', 'profeel', 'building_stock_profeel_std.csv'), index_col='Housing type')
sdes = pd.read_csv(os.path.join('data', 'derived', 'building_stock', 'building_stock_sdes2018_aggregated.csv'), index_col=['Housing type', 'Energy performance', 'Heating energy'])

profeel = profeel.rename(columns={'calculated_epc':'Energy performance'})
profeel['Heating energy'] = profeel['Heating system'].str.split('-').str[0]
profeel = profeel.set_index(['Heating energy', 'Energy performance'], append=True)


In [ ]:
merged = profeel.merge(sdes, how='inner', on=['Housing type', 'Energy performance', 'Heating energy'])
# Adding a row specific index level
r_position = pd.Series(range(0,merged.shape[0]), name='r_position')
merged = merged.set_index(r_position, append=True)

# Weight of each lines in terms of sdes's sizes (y) among a profeel archetype uniquely define by its class label
merged = merged.set_index('Class', append=True)
merged['Stock buildings_y'] = merged['Stock buildings_y'].T / merged.groupby(level='Class')['Stock buildings_y'].sum()

# Multiplying the weights to the profeel's stocks (x)
merged['Stock buildings'] = (merged['Stock buildings_y'].T * merged['Stock buildings_x']).T
cleaned = merged.reset_index(('Energy performance', 'r_position', 'Class', 'Heating energy'))
cleaned = cleaned.drop(labels=['Stock buildings_x', 'Stock buildings_y', 'Energy performance', 'r_position', 'Class', 'Heating energy'], axis=1)
cleaned


In [ ]:
# Checks
print("profeel building stock: {}".format(profeel['Stock buildings'].sum()))
print("sdes building stock: {}".format(sdes['Stock buildings'].sum()))
print("The number of new lines for each profeel archetypes: {}".format(merged.shape[0]/profeel.shape[0]))
print("merged building stock: {}".format(merged['Stock buildings_x'].sum()/(merged.shape[0]/profeel.shape[0])))
print("output building stock: {}".format(cleaned['Stock buildings'].sum()))
print("building stock loss: {}".format(profeel['Stock buildings'].sum() - cleaned['Stock buildings'].sum()))


In [ ]:
os.makedirs(os.path.join('data', 'derived', 'profeel'), exist_ok=True)
cleaned.to_csv(os.path.join('data', 'derived', 'profeel', 'building_stock_profeel_final.csv'))
